<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-03-prompt-engineering/lesson-3.2-few-shot-and-in-context/notebooks/exercises-3.2.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 3.2: Few-Shot and In-Context Learning - Practice Exercises

**Netsetos GenAI Course v2.0** | Module 3 | 8 exercises: shot comparison, format design, shot count, ordering effects, dynamic selection, and a production mini-project.

---

In [ ]:
!pip install google-generativeai scikit-learn -q
import google.generativeai as genai
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# genai.configure(api_key='YOUR_KEY')
model = genai.GenerativeModel('gemini-2.0-flash')

def ask(prompt, temp=0.0):
    r = model.generate_content(prompt, generation_config={'temperature': temp})
    return r.text.strip()


---
## Exercise 1 (Easy): Zero-Shot vs Few-Shot
Measure how examples improve a support-ticket classifier.

In [ ]:
tickets = [
    ('Where is my order #456?', 'delivery'),
    ('I want to return this shirt', 'return'),
    ('The app crashes after login', 'technical'),
    ('Can I change my account email?', 'account'),
    ('Refund still not received after 7 days', 'return'),
]

zero_template = '''Classify the support ticket into one of these labels:
delivery, return, technical, account
Output ONLY the label.
Ticket: "{text}"'''

few_template = '''Classify the support ticket.
Output ONLY one label: delivery, return, technical, account.

Ticket: "My package never arrived" -> delivery
Ticket: "I want my money back" -> return
Ticket: "The dashboard shows a blank screen" -> technical
Ticket: "I forgot my password" -> account

Ticket: "{text}" ->'''

zero_correct = 0
few_correct = 0
for text, label in tickets:
    # z = ask(zero_template.format(text=text), temp=0).lower()
    # f = ask(few_template.format(text=text), temp=0).lower()
    # zero_correct += int(label in z)
    # few_correct += int(label in f)
    pass

# print(f'Zero-shot: {zero_correct}/{len(tickets)}')
# print(f'Few-shot:  {few_correct}/{len(tickets)}')


---
## Exercise 2 (Easy): Prompt Format Comparison
Compare arrow, XML, and JSON example formatting.

In [ ]:
review = 'The phone looks great but battery life is weak.'

arrow_prompt = '''Review: "Amazing camera, very happy" -> positive
Review: "Poor build quality, disappointed" -> negative
Review: "It is okay for the price" -> neutral
Review: "{review}" ->'''

xml_prompt = '''<example><review>Amazing camera, very happy</review><sentiment>positive</sentiment></example>
<example><review>Poor build quality, disappointed</review><sentiment>negative</sentiment></example>
<example><review>It is okay for the price</review><sentiment>neutral</sentiment></example>
<review>{review}</review><sentiment>'''

json_prompt = '''Examples:
{"review":"Amazing camera, very happy","sentiment":"positive"}
{"review":"Poor build quality, disappointed","sentiment":"negative"}
{"review":"It is okay for the price","sentiment":"neutral"}
Input: {"review":"{review}"}
Output: '''

for name, prompt in [('arrow', arrow_prompt), ('xml', xml_prompt), ('json', json_prompt)]:
    # responses = [ask(prompt.format(review=review), temp=0.3) for _ in range(3)]
    # print(name, responses)
    pass


---
## Exercise 3 (Medium): How Many Examples?
Find the accuracy-vs-cost sweet spot.

In [ ]:
example_pool = [
    ('Great camera and smooth performance', 'positive'),
    ('Stopped working after one week', 'negative'),
    ('Average laptop, does the job', 'neutral'),
    ('Excellent display quality', 'positive'),
    ('Battery drains too fast', 'negative'),
    ('Fine for casual use', 'neutral'),
    ('Very happy with the purchase', 'positive'),
    ('Poor speaker quality', 'negative')
]

test_reviews = [
    ('Amazing value for the money', 'positive'),
    ('Build quality feels cheap', 'negative'),
    ('Nothing special but acceptable', 'neutral')
]

def build_prompt(examples, review):
    if not examples:
        return f'Classify sentiment as positive, negative, or neutral. Output ONLY the label. Review: {review}'
    demo = '\n'.join([f'Review: "{r}" -> {s}' for r, s in examples])
    return f'Classify sentiment. Output ONLY the label.\n\n{demo}\n\nReview: "{review}" ->'

for shot_count in [0, 1, 3, 5, 8]:
    chosen = example_pool[:shot_count]
    # correct = 0
    # for review, label in test_reviews:
    #     pred = ask(build_prompt(chosen, review), temp=0).lower()
    #     correct += int(label in pred)
    # print(f'{shot_count}-shot: {correct}/{len(test_reviews)}')
    pass


---
## Exercise 4 (Medium): Ordering Effects
Use the same examples in different orders and inspect the result drift.

In [ ]:
examples = [
    ('Brilliant quality, loved it', 'positive'),
    ('Terrible quality, regret buying', 'negative'),
    ('Good but not amazing', 'neutral')
]

orders = {
    'pos-last': [examples[1], examples[2], examples[0]],
    'neg-last': [examples[0], examples[2], examples[1]],
    'neutral-last': [examples[0], examples[1], examples[2]],
}

ambiguous_review = 'Looks premium, but the battery is disappointing.'

for name, ordered in orders.items():
    prompt = 'Classify sentiment. Output only the label.\n\n'
    prompt += '\n'.join([f'Review: "{r}" -> {s}' for r, s in ordered])
    prompt += f'\n\nReview: "{ambiguous_review}" ->'
    # print(name, ask(prompt, temp=0))
    pass


---
## Exercise 5 (Medium): Static vs Dynamic Example Selection
Use TF-IDF similarity as a lightweight stand-in for embedding retrieval.

In [ ]:
bank = [
    ('My package is late again', 'delivery'),
    ('Tracking says delivered but I never got it', 'delivery'),
    ('I want to return this damaged item', 'return'),
    ('Refund has not arrived yet', 'return'),
    ('The website throws a 500 error', 'technical'),
    ('App crashes on startup', 'technical'),
    ('I cannot log into my account', 'account'),
    ('Need to reset my password', 'account')
]

texts = [x[0] for x in bank]
vectorizer = TfidfVectorizer()
matrix = vectorizer.fit_transform(texts)

query = 'My order still has not arrived'
query_vec = vectorizer.transform([query])
scores = cosine_similarity(query_vec, matrix)[0]
top_idx = np.argsort(scores)[::-1][:3]
selected = [bank[i] for i in top_idx]
selected


---
## Exercise 6 (Hard): Coverage Beats Quantity
Compare a repetitive example bank with a diverse one.

In [ ]:
repetitive = [
    ('My package is late', 'delivery'),
    ('Order is delayed', 'delivery'),
    ('Shipment not received', 'delivery')
]

diverse = [
    ('My package is late', 'delivery'),
    ('I need a refund', 'return'),
    ('The app keeps crashing', 'technical')
]

target = 'The refund for my damaged order still has not come through.'

def classify_with_examples(examples, target_text):
    demo = '\n'.join([f'Ticket: "{r}" -> {label}' for r, label in examples])
    prompt = f'Classify the support ticket as delivery, return, technical, or account. Output only the label.\n\n{demo}\n\nTicket: "{target_text}" ->'
    # return ask(prompt, temp=0)
    return 'run me'

print('repetitive:', classify_with_examples(repetitive, target))
print('diverse:', classify_with_examples(diverse, target))


---
## Exercise 7 (Hard): Dynamic Few-Shot Pipeline
Build a reusable classifier that retrieves examples before prompting.

In [ ]:
def select_examples(query, bank, k=3):
    vectorizer = TfidfVectorizer()
    texts = [x[0] for x in bank]
    matrix = vectorizer.fit_transform(texts)
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, matrix)[0]
    top_idx = np.argsort(scores)[::-1][:k]
    return [bank[i] for i in top_idx]

def classify_ticket(query, bank, k=3):
    examples = select_examples(query, bank, k=k)
    demo = '\n'.join([f'Ticket: "{x}" -> {y}' for x, y in examples])
    prompt = f'Classify the ticket as delivery, return, technical, or account. Output ONLY the label.\n\n{demo}\n\nTicket: "{query}" ->'
    # return ask(prompt, temp=0), examples
    return 'label', examples

pred, chosen = classify_ticket('Need a refund because the item arrived broken', bank)
print(pred)
print(chosen)


---
## Exercise 8 (Hard): Production Review
Think like an engineer, not just a prompt writer.

In [ ]:
questions = [
    'Which failure cases should be added back into the example bank?',
    'What metrics will you track: accuracy, latency, token cost, or all three?',
    'At what point would you move from static examples to dynamic retrieval?',
    'When would RAG or fine-tuning beat few-shot prompting for this task?'
]

for q in questions:
    print('-', q)
